In [1]:
%pip install selenium undetected-chromedriver "setuptools<70.0.0" numpy pandas packaging

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys, types, time, os, re
import numpy as np
import pandas as pd

# Patch distutils (bị remove ở Python 3.12+)
# Giữ đầy đủ vstring để undetected_chromedriver truy cập trực tiếp
class MinimalLooseVersion:
    def __init__(self, v):
        self.vstring = str(v)
        self.version = [int(x) if x.isdigit() else x for x in self.vstring.split('.')]
    def __lt__(self, o): return self.version < o.version
    def __gt__(self, o): return self.version > o.version
    def __eq__(self, o): return self.version == o.version
    def __ge__(self, o): return self.version >= o.version
    def __le__(self, o): return self.version <= o.version

distutils = types.ModuleType('distutils')
distutils.version = types.ModuleType('distutils.version')
distutils.version.LooseVersion = MinimalLooseVersion
sys.modules['distutils'] = distutils
sys.modules['distutils.version'] = distutils.version

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import selenium.webdriver.support.expected_conditions as EC
import undetected_chromedriver as uc

In [3]:
def create_driver():
    """Tạo driver với cấu hình tối giản, tránh dùng experimental_option không tương thích uc."""
    options = uc.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    # Tắt ảnh/css qua argument thay vì prefs (tương thích với undetected_chromedriver)
    options.add_argument("--blink-settings=imagesEnabled=false")
    options.page_load_strategy = 'normal'
    return uc.Chrome(options=options, version_main=148)

In [4]:
def wait_for_cloudflare(driver, timeout=30):
    """Chờ vượt qua màn hình Cloudflare. Trả về True nếu thành công."""
    try:
        WebDriverWait(driver, timeout).until_not(
            EC.title_contains("Just a moment")
        )
        time.sleep(1.5)
        return True
    except Exception:
        print("❌ Cloudflare timeout.")
        return False

In [5]:
def get_hrefs_from_page(driver, url):
    """Lấy danh sách href từ 1 trang listing. Trả về list rỗng nếu lỗi."""
    driver.get(url)
    if not wait_for_cloudflare(driver):
        raise RuntimeError("Cloudflare block — cần retry")

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "js__product-link-for-product-id"))
        )
    except Exception:
        print(f"  ⚠️ Không thấy listing. Title: {driver.title}")
        return []  # Trang thật sự trống (không phải Cloudflare) → bỏ qua bình thường

    elements = driver.find_elements(
        By.CSS_SELECTOR, ".re__srp-list .js__product-link-for-product-id"
    )
    hrefs = []
    for el in elements:
        href = el.get_attribute("href")
        if not href:
            continue
        if "?" not in href:
            href += "?vrs=1"
        elif "vrs=1" not in href:
            href += "&vrs=1"
        hrefs.append(href)
    return hrefs

In [6]:
BASE_URL       = "https://batdongsan.com.vn/ban-nha-dat-tp-hcm"
QUERY          = "?avrs=1"
TARGET_SAMPLES = 10  # ⬅️ Đổi số này để kiểm soát đầu ra
MAX_PAGE_RETRY = 3      # Số lần retry tối đa mỗi trang khi driver lỗi

href_driver  = [create_driver()]
all_hrefs    = []
seen         = set()
page         = 1
page_retries = 0

while len(all_hrefs) < TARGET_SAMPLES:
    url = f"{BASE_URL}{f'/p{page}' if page > 1 else ''}{QUERY}"
    print(f"📄 Trang {page} | {len(all_hrefs)}/{TARGET_SAMPLES}: {url}")

    try:
        hrefs = get_hrefs_from_page(href_driver[0], url)
        page_retries = 0  # reset counter khi thành công
    except Exception as e:
        page_retries += 1
        print(f"  ❌ Driver lỗi trang {page} (lần {page_retries}/{MAX_PAGE_RETRY}): {e}")
        try: href_driver[0].quit()
        except Exception: pass
        href_driver[0] = create_driver()
        if page_retries >= MAX_PAGE_RETRY:
            print(f"  ⛔ Bỏ qua trang {page} sau {MAX_PAGE_RETRY} lần thất bại.")
            page += 1
            page_retries = 0
        continue

    for href in hrefs:
        if href not in seen and len(all_hrefs) < TARGET_SAMPLES:
            seen.add(href)
            all_hrefs.append(href)

    href_driver[0].execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(0.5)
    next_btn = href_driver[0].find_elements(
        By.CSS_SELECTOR, "a.re__pagination-icon:not(.re__pagination-icon--no-effect)"
    )
    if not next_btn:
        print("🏁 Hết trang.")
        break

    page += 1
    time.sleep(1.5)

href_driver[0].quit()
print(f"\n✅ Tổng: {len(all_hrefs)} links từ {page} trang")

with open("hrefs.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(all_hrefs))
print("💾 Đã lưu hrefs.txt")

📄 Trang 1 | 0/10: https://batdongsan.com.vn/ban-nha-dat-tp-hcm?avrs=1

✅ Tổng: 10 links từ 2 trang
💾 Đã lưu hrefs.txt


In [7]:
def extract_property_info(driver, url):
    """
    Cào thông tin chi tiết 1 BĐS.
    Trả về dict với đầy đủ key; giá trị NaN nếu không lấy được.
    """
    info = {
        'Link'                   : url,
        'Địa chỉ'                : np.nan,
        'Khoảng giá'             : np.nan,
        'Diện tích'              : np.nan,
        'Số phòng ngủ'           : np.nan,
        'Số phòng tắm, vệ sinh'  : np.nan,
        'Số tầng'                : np.nan,
        'Hướng nhà'              : np.nan,
        'Hướng ban công'         : np.nan,
        'Đường vào'              : np.nan,
        'Pháp lý'                : np.nan,
        'Nội thất'               : np.nan,
        'Latitude'               : np.nan,
        'Longitude'              : np.nan,
        'Mô tả'                  : np.nan,
    }

    try:
        driver.get(url)
        if not wait_for_cloudflare(driver):
            return info

        # Chờ trang chính load xong (dùng Địa chỉ làm anchor)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "re__pr-other-info-display"))
        )
    except Exception:
        # Trang lỗi — vẫn cố lấy Mô tả nếu có thể
        try:
            info['Mô tả'] = driver.find_element(
                By.CLASS_NAME, 're__section-body.re__detail-content'
            ).text.strip() or np.nan
        except Exception:
            pass
        return info

    # 1. Địa chỉ
    try:
        info['Địa chỉ'] = driver.find_element(
            By.CLASS_NAME, 're__address-line-1'
        ).text.strip() or np.nan
    except Exception:
        pass

    # 2. Tọa độ từ iframe bản đồ
    try:
        data_src = driver.find_element(
            By.CSS_SELECTOR, 'iframe.lazyload'
        ).get_attribute('data-src')
        coords = data_src.split('q=')[1].split('&')[0].split(',')
        info['Latitude']  = float(coords[0].strip())
        info['Longitude'] = float(coords[1].strip())
    except Exception:
        pass

    # 3. Các thông số specs (Khoảng giá, Diện tích, PN, WC, ...)
    try:
        specs = driver.find_elements(
            By.CSS_SELECTOR,
            '.re__pr-other-info-display .re__pr-specs-content-item'
        )
        for spec in specs:
            try:
                title = spec.find_element(
                    By.CLASS_NAME, 're__pr-specs-content-item-title'
                ).text.strip()
                value = spec.find_element(
                    By.CLASS_NAME, 're__pr-specs-content-item-value'
                ).text.strip()
                if title in info and value:
                    info[title] = value
            except Exception:
                continue
    except Exception:
        pass

    # 4. Mô tả
    try:
        mo_ta = driver.find_element(
            By.CLASS_NAME, 're__section-body.re__detail-content'
        ).text.strip()
        info['Mô tả'] = mo_ta if mo_ta else np.nan
    except Exception:
        pass

    return info

In [8]:
with open("hrefs.txt", "r", encoding="utf-8") as f:
    urls = [u.strip() for u in f.read().splitlines() if u.strip()]

CSV_FILE    = r'C:\Users\ADMIN\OneDrive\Desktop\Thu thập và tiền xử lí dữ liệu bất động sản phục vụ cho bài toán phân loại theo giá\data\Raw\data_bds.csv'
BATCH_SIZE  = 50
RETRY_DELAY = 2.0
PAGE_DELAY  = 0.3
MAX_RETRIES = 2

processed = set()
batch     = []

if os.path.exists(CSV_FILE):
    os.remove(CSV_FILE)
    print(f"🗑️ Đã xóa file cũ: {CSV_FILE}")

# ── Dùng list 1 phần tử để driver có thể được update trong mọi scope ─────────
d = [create_driver()]

def get_driver():
    return d[0]

def replace_driver():
    try:
        d[0].quit()
    except Exception:
        pass
    d[0] = create_driver()
    print("🔄 Đã tạo lại driver")

def flush_batch(batch, csv_file):
    if not batch:
        return
    df_batch = pd.DataFrame(batch)
    is_first = not os.path.exists(csv_file)
    df_batch.to_csv(csv_file, mode='a', header=is_first, index=False, encoding='utf-8-sig')
    print(f"💾 Đã lưu {len(batch)} records")
    batch.clear()

def crawl_with_retry(url):
    """Cào 1 URL, retry tối đa MAX_RETRIES lần. Luôn dùng driver hiện tại."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            info = extract_property_info(get_driver(), url)
            if pd.isna(info.get('Địa chỉ')) and attempt < MAX_RETRIES:
                print(f"  ⚠️ Địa chỉ NaN — retry {attempt}/{MAX_RETRIES}...")
                time.sleep(RETRY_DELAY)
                continue
            return info
        except Exception as e:
            print(f"  ❌ Exception attempt {attempt}: {e}")
            replace_driver()
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
    # Hết retry — trả về dict tối thiểu để không mất dòng
    print(f"  ⛔ Bỏ qua sau {MAX_RETRIES} lần thất bại: {url}")
    return {'Link': url}

# ── Vòng lặp chính ────────────────────────────────────────────────────────────
for idx, url in enumerate(urls, 1):
    if url in processed:
        continue
    processed.add(url)

    print(f"[{idx}/{len(urls)}] {url}")
    info = crawl_with_retry(url)
    batch.append(info)

    if len(batch) >= BATCH_SIZE:
        flush_batch(batch, CSV_FILE)

    time.sleep(PAGE_DELAY)

flush_batch(batch, CSV_FILE)
get_driver().quit()
print(f"\n✅ Hoàn thành. File: {CSV_FILE}")

[1/10] https://batdongsan.com.vn/ban-nha-biet-thu-lien-ke-xa-xuan-thoi-son-vinhomes-saigon-park/pho-gia-tot-can-ep-giai-oan-1-ang-nhan-booking-uu-tien-100-trieu-cho-co-hoan-lai-pr45747147?vrs=1
[2/10] https://batdongsan.com.vn/ban-nha-biet-thu-lien-ke-xa-xuan-thoi-son-vinhomes-saigon-park/booking-suat-noi-bo-hoc-mon-gia-du-kien-5-ty-cho-pho-he-truc-tiep-c-t-pr45810834?vrs=1
[3/10] https://batdongsan.com.vn/ban-nha-biet-thu-lien-ke-xa-xuan-thoi-son-vinhomes-saigon-park/suat-ngoai-giao-ck-18-5-vinhome-hoc-mon-gia-chi-5-ty-can-he-truc-tiep-c-t-pr45656262?vrs=1
[4/10] https://batdongsan.com.vn/ban-nha-rieng-duong-quang-trung-phuong-8-12-67/dien-tich-khung-1002-at-ngang-6-5m-co-5-phong-ngu-rong-hon-6ty-x-uong-f8-go-vap-pr45815861?vrs=1
[5/10] https://batdongsan.com.vn/ban-nha-biet-thu-lien-ke-xa-xuan-thoi-son-vinhomes-saigon-park/booking-som-chon-can-ep-on-au-co-hoi-tang-gia-sai-gon-lh-em-khoa-tu-van-24-7-pr45804194?vrs=1
[6/10] https://batdongsan.com.vn/ban-nha-biet-thu-lien-ke-xa-xuan-tho

In [ ]:
import re as _re

def extract_from_description(desc):
    """
    Trích xuất thuộc tính từ mô tả rao bán BĐS.
    Trả về dict; NaN nếu không tìm thấy.
    """
    if not isinstance(desc, str) or not desc.strip():
        return {}

    txt = desc.lower()   # đổi tên biến tránh shadow `d` (driver container)
    result = {}

    # --- Diện tích ---
    m = _re.search(r'\((\d+[,.]?\d*)\s*m[²2]', txt)
    if not m:
        m = _re.search(r'di[eệ]n t[ií]ch(?:\s+(?:đ[aấ]t|khu[ôo]n đ[aấ]t|thực tế))?[:\s]*(\d+[,.]?\d*)\s*m', txt)
    result['DT đất (desc)'] = m.group(1).replace(',', '.') if m else np.nan

    m = _re.search(r'di[eệ]n t[ií]ch\s*(?:xây dựng|sàn|sd|sử dụng)[:\s]*(\d+[,.]?\d*)\s*m', txt)
    result['DT xây dựng (desc)'] = m.group(1).replace(',', '.') if m else np.nan

    m = _re.search(r'(?:ngang[:\s]*)?\(?([,\d\.]+)\s*[x×]\s*([,\d\.]+)\s*m?\)?', txt)
    result['Ngang (desc)'] = m.group(1).replace(',', '.') if m else np.nan
    result['Dài (desc)']   = m.group(2).replace(',', '.') if m else np.nan

    # --- Số tầng ---
    m = _re.search(r'(\d+[,.]?\d*)\s*t[aầ]ng', txt)
    if m:
        result['Số tầng (desc)'] = m.group(1)
    else:
        m_tret = _re.search(r'(\d+)\s*tr[eệ]t', txt)
        m_lau  = _re.search(r'(\d+)\s*l[aầ]u', txt)
        if m_tret and m_lau:
            result['Số tầng (desc)'] = str(int(m_tret.group(1)) + int(m_lau.group(1)))
        elif m_tret:
            result['Số tầng (desc)'] = m_tret.group(1)
        elif m_lau:
            result['Số tầng (desc)'] = str(int(m_lau.group(1)) + 1)
        else:
            result['Số tầng (desc)'] = np.nan

    # --- Số PN / WC ---
    m = _re.search(r'(\d+)\s*(?:phòng[\s-]?ngủ|pn\b|bedroom)', txt)
    result['Số PN (desc)'] = m.group(1) if m else np.nan

    m = _re.search(r'(\d+)\s*(?:toilet|wc\b|nhà vệ sinh|phòng tắm\b)', txt)
    result['Số WC (desc)'] = m.group(1) if m else np.nan

    # --- Hướng ---
    HUONG = r'(đông[\s-]?nam|đông[\s-]?bắc|tây[\s-]?nam|tây[\s-]?bắc|đông|tây|nam|bắc)'
    m = _re.search(r'hướng(?:\s+(?:cửa chính|nhà|chính))?[:\s]*' + HUONG, txt)
    result['Hướng nhà (desc)'] = m.group(1).title() if m else np.nan

    m = _re.search(r'hướng ban công[:\s]*' + HUONG, txt)
    result['Hướng BC (desc)'] = m.group(1).title() if m else np.nan

    # --- Pháp lý ---
    phap_ly_map = {
        'sổ hồng': 'Sổ hồng', 'sổ đỏ': 'Sổ đỏ', 'shr': 'Sổ hồng',
        'hợp đồng mua bán': 'HĐMB', 'hdmb': 'HĐMB',
        'giấy tờ tay': 'Giấy tay', 'giấy tay': 'Giấy tay',
        'chờ sổ': 'Chờ sổ', 'đang làm sổ': 'Chờ sổ',
    }
    result['Pháp lý (desc)'] = np.nan
    for k, v in phap_ly_map.items():
        if k in txt:
            result['Pháp lý (desc)'] = v
            break

    # --- Nội thất ---
    if _re.search(r'full n[oộ]i th[aấ]t|đầy đủ n[oộ]i th[aấ]t|n[oộ]i th[aấ]t cao c[aấ]p', txt):
        result['Nội thất (desc)'] = 'Đầy đủ'
    elif _re.search(r'n[oộ]i th[aấ]t c[oơ] b[aả]n|c[oơ] b[aả]n', txt):
        result['Nội thất (desc)'] = 'Cơ bản'
    elif _re.search(r'không n[oộ]i th[aấ]t|bàn giao thô|nhà thô', txt):
        result['Nội thất (desc)'] = 'Không'
    else:
        result['Nội thất (desc)'] = np.nan

    return result


# ── Áp dụng lên toàn bộ DataFrame ────────────────────────────────────────────
df = pd.read_csv(r'C:\Users\ADMIN\OneDrive\Desktop\Thu thập và tiền xử lí dữ liệu bất động sản phục vụ cho bài toán phân loại theo giá\data\Raw\data_bds.csv', encoding='utf-8-sig')
desc_df = pd.DataFrame(df['Mô tả'].apply(extract_from_description).tolist())

overlap = [c for c in desc_df.columns if c in df.columns]
desc_df = desc_df.drop(columns=overlap)
df = pd.concat([df, desc_df], axis=1)

fill_map = {
    'Diện tích'            : 'DT đất (desc)',
    'Số tầng'              : 'Số tầng (desc)',
    'Số phòng ngủ'         : 'Số PN (desc)',
    'Số phòng tắm, vệ sinh': 'Số WC (desc)',
    'Hướng nhà'            : 'Hướng nhà (desc)',
    'Hướng ban công'       : 'Hướng BC (desc)',
    'Pháp lý'              : 'Pháp lý (desc)',
    'Nội thất'             : 'Nội thất (desc)',
    'Đường vào'            : 'Đường vào (desc)',
}
for col_goc, col_desc in fill_map.items():
    if col_goc in df.columns and col_desc in df.columns:
        df[col_goc] = df[col_goc].fillna(df[col_desc])

if 'Loại đường vào (desc)' in df.columns:
    df.rename(columns={'Loại đường vào (desc)': 'Loại đường vào'}, inplace=True)

df.drop(columns=[c for c in df.columns if c.endswith('(desc)')], inplace=True)

df.to_csv(r'C:\Users\ADMIN\OneDrive\Desktop\Thu thập và tiền xử lí dữ liệu bất động sản phục vụ cho bài toán phân loại theo giá\data\Raw\data_bds_NaNFill.csv', index=False, encoding='utf-8-sig')
print(f"✅ Đã xuất data_bds_expanded.csv — shape: {df.shape}")
print(df.isnull().mean().mul(100).round(1).to_string())

✅ Đã xuất data_bds_expanded.csv — shape: (10, 15)
Link                      0.0
Địa chỉ                   0.0
Khoảng giá                0.0
Diện tích                 0.0
Số phòng ngủ             50.0
Số phòng tắm, vệ sinh    50.0
Số tầng                  30.0
Hướng nhà                60.0
Hướng ban công           80.0
Đường vào                80.0
Pháp lý                  50.0
Nội thất                 40.0
Latitude                  0.0
Longitude                 0.0
Mô tả                     0.0
